# Trabajo de Laboratorio N° 2: Sistemas Inteligentes
## NLP – Representaciones Vectoriales – RAG – Agentes Basados en LLMs
* **Dominio Técnico:** GitHub REST API (Especificación v3 / Versión 2022-11-28)
* **Cátedra:** Inteligencia Artificial – FCEFyN
* **Modelo Local Utilizado:** `qwen2.5:1.5b` (Ollama local en http://localhost:11434)
* **Alcance de este Notebook:** 
  * **Fase 2:** Construcción del Corpus, Estrategia de Chunking, Embeddings e Indexación Vectorial (FAISS) y Validación de Búsqueda Semántica (*Retrieval*).
  * **Fase 3 (Parte 1 - Conexión e Integración):** Vinculación del motor de recuperación vectorial con el LLM local mediante LangChain Expression Language (LCEL).
  * **Fase 3 (Parte 2 - Lógica de Agente y Herramientas):** Orquestación de un Agente Interactivo (ReAct) con herramientas de RAG y consultas externas en vivo a la API de GitHub.

## 1. Instalación e Importación de Dependencias
Instalamos las librerías necesarias para el procesamiento de texto, base vectorial FAISS, clientes de LLMs locales y componentes de agentes interactivos de LangChain.

In [ ]:
# Instalación de librerías requeridas (ejecutar en Google Colab o entorno nuevo)
!pip install -q langchain langchain-community langchain-openai langchain-ollama langchain-text-splitters faiss-cpu sentence-transformers requests

In [ ]:
import os
import glob
import json
import requests
from typing import List, Dict, Any

# LangChain y componentes de RAG y Agentes
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

print("✅ Librerías importadas correctamente.")

## 2. Fase 2: Carga del Corpus Técnico y Segmentación (Chunking)
Cargamos los documentos técnicos formalizados en `./corpus`:
1. `01_autenticacion_y_rate_limits.md`
2. `02_repositorios_y_ramas.md`
3. `03_issues_y_etiquetas.md`
4. `04_pull_requests_y_revisiones.md`

Aplicamos `RecursiveCharacterTextSplitter` para segmentar los documentos preservando la coherencia semántica de encabezados, tablas y bloques de código de endpoints.

In [ ]:
# 1. Localización de los archivos del corpus
CORPUS_DIR = "./corpus"
md_files = sorted(glob.glob(os.path.join(CORPUS_DIR, "*.md")))
tech_files = [f for f in md_files if not os.path.basename(f).lower().startswith("readme")]

print(f"📄 Archivos técnicos encontrados ({len(tech_files)}):")
for f in tech_files:
    print(f"  - {f}")

# 2. Carga con TextLoader
raw_documents = []
for filepath in tech_files:
    loader = TextLoader(filepath, encoding="utf-8")
    raw_documents.extend(loader.load())

print(f"\n✅ Total de documentos originales cargados: {len(raw_documents)}")

# 3. Segmentación (Chunking) semántica
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=120,
    separators=["\n## ", "\n### ", "\n\n", "\n", " "]
)

chunks = text_splitter.split_documents(raw_documents)
print(f"✅ Documentos divididos en {len(chunks)} chunks.")

# 4. Inspección de un chunk representativo
print("\n🔍 Ejemplo de chunk generado:")
print(f"Fuente: {chunks[0].metadata['source']}")
print("-" * 50)
print(chunks[0].page_content[:350] + "...")

## 3. Fase 2: Generación de Embeddings e Indexación Vectorial (FAISS)
Convertimos cada chunk en una representación vectorial utilizando el modelo de Hugging Face (`sentence-transformers/all-MiniLM-L6-v2`) y los almacenamos en un índice vectorial **FAISS** optimizado para búsquedas por similitud de cosenos.

In [ ]:
# Selección del modelo de embeddings
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
print(f"⏳ Inicializando modelo de embeddings: {EMBEDDING_MODEL_NAME}...")

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

# Construcción de la base vectorial FAISS
print("⏳ Construyendo índice vectorial FAISS a partir de los chunks...")
vectorstore = FAISS.from_documents(documents=chunks, embedding=embedding_model)

# Persistencia local opcional
vectorstore.save_local("faiss_github_api_index")
print("✅ Índice vectorial FAISS creado y guardado en disco exitosamente.")

## 4. Fase 2: Motor de Recuperación (Retrieval) y Validación Semántica
Configuramos el `retriever` para recuperar los 3 fragmentos más cercanos a la consulta del usuario y testeamos la recuperación de información técnica pura antes de pasársela al LLM.

In [ ]:
# Configuración del Retriever (Top-3 por similitud de cosenos)
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

def probar_retrieval(consulta: str):
    print(f"\n❓ Consulta de prueba: '{consulta}'")
    docs = retriever.invoke(consulta)
    print(f"📄 Fragmentos recuperados: {len(docs)}")
    for i, doc in enumerate(docs, 1):
        origen = os.path.basename(doc.metadata.get('source', 'desconocido'))
        print(f"\n  [Chunk {i} | Fuente: {origen}]")
        print("  " + doc.page_content.strip()[:250].replace('\n', '\n  ') + "...")
    print("=" * 60)

# Validaciones semánticas del motor de recuperación
probar_retrieval("¿Cuáles son los encabezados obligatorios y qué pasa si no se envía User-Agent?")
probar_retrieval("¿Cómo crear un issue y qué parámetros son requeridos?")
probar_retrieval("¿Cuáles son los métodos disponibles para hacer merge de un Pull Request?")

## 5. Fase 3: Conexión e Integración (Vincular Retriever con LLM Local Ollama)
En este bloque completamos la etapa de **Conexión e Integración** requerida por la Fase 3 de la consigna:
> *"Vincular el motor de recuperación vectorial con un modelo generativo de lenguaje (LLM de acceso abierto local o a través de una API)."*

Utilizamos el modelo **`qwen2.5:1.5b`** ejecutado de manera 100% local a través de **Ollama** (`http://localhost:11434/v1`).

### Arquitectura de Integración (RAG Pipeline LCEL):
```text
           [ Consulta del Usuario ]
                      │
        ┌─────────────┴─────────────┐
        ▼                           ▼
[ Retriever (FAISS) ]     [ Prompt Template ]
        │ (Top-3 Chunks)            │
        └─────────────►┬◄───────────┘
                       │ Contexto + Pregunta
                       ▼
          [ LLM Local (qwen2.5:1.5b) ]
                       │
                       ▼
         [ Respuesta Fundamentada ]
```

In [ ]:
# Conexión con el LLM local mediante Ollama (qwen2.5:1.5b)
# Ollama expone un endpoint compatible con OpenAI en http://localhost:11434/v1
OLLAMA_BASE_URL = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434/v1")
MODEL_NAME = "qwen2.5:1.5b"

llm = ChatOpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key="ollama",  # Requerido por el cliente; Ollama no valida credencial
    model=MODEL_NAME,
    temperature=0.1
)

print(f"✅ Conexión con el LLM local establecida correctamente ({MODEL_NAME} en {OLLAMA_BASE_URL}).")

In [ ]:
# Formateador de contexto con trazabilidad de fuentes
def formatear_contexto(docs: List[Any]) -> str:
    bloques = []
    for i, doc in enumerate(docs, 1):
        fuente = os.path.basename(doc.metadata.get("source", "desconocido"))
        bloques.append(f"--- [FRAGMENTO {i} | FUENTE: {fuente}] ---\n{doc.page_content.strip()}")
    return "\n\n".join(bloques)

# Prompt del Sistema y Plantilla para RAG
template_rag = """Eres un Asistente Técnico Senior especializado en la API REST de GitHub (v3).
Tu tarea es responder preguntas técnicas con máxima precisión basándote ÚNICAMENTE en la documentación oficial provista en el CONTEXTO.

Instrucciones obligatorias:
1. Detalla métodos HTTP, rutas completas, códigos de estado, parámetros y cabeceras según corresponda.
2. Incluye siempre la cita de la fuente utilizada (por ejemplo: [Fuente: 01_autenticacion_y_rate_limits.md]).
3. Si la información no se encuentra en el contexto provisto, responde exactamente: "La documentación provista en el corpus no contiene información suficiente para responder a esta consulta." No agregues suposiciones.

CONTEXTO TÉCNICO OFICIAL:
{contexto}

PREGUNTA DEL USUARIO:
{pregunta}

RESPUESTA TÉCNICA FUNDAMENTADA:"""

prompt_rag = ChatPromptTemplate.from_template(template_rag)

# Pipeline LCEL (LangChain Expression Language) para Conexión e Integración
cadena_rag = (
    {
        "contexto": retriever | formatear_contexto,
        "pregunta": RunnablePassthrough()
    }
    | prompt_rag
    | llm
    | StrOutputParser()
)

print("✅ Pipeline RAG (Retriever + Prompt + LLM) conectado e integrado.")

## 6. Validación de la Cadena RAG Básica (LCEL)
Ejecutamos consultas técnicas para comprobar la integración de la cadena completa de RAG puro (recuperación + síntesis sin alucinaciones) antes de orquestar el agente autónomo.

In [ ]:
def consultar_asistente_rag(pregunta: str) -> str:
    print("=" * 70)
    print(f"🧑‍💻 CONSULTA: {pregunta}")
    print("=" * 70)
    respuesta = cadena_rag.invoke(pregunta)
    print(f"🤖 RESPUESTA:\n{respuesta}\n")
    return respuesta

# 1. Consulta sobre Autenticación y Cabeceras
r1 = consultar_asistente_rag(
    "¿Cuáles son los encabezados HTTP obligatorios para hacer peticiones a la API de GitHub y qué ocurre si no envío el User-Agent?"
)

# 2. Consulta sobre Pull Requests y Parámetros
r2 = consultar_asistente_rag(
    "¿Qué método y endpoint debo usar para crear un Pull Request y cuáles son los campos obligatorios del cuerpo JSON?"
)

# 3. Consulta sobre Rate Limiting y Cuotas
r3 = consultar_asistente_rag(
    "¿Cuál es el límite de solicitudes por hora para usuarios no autenticados frente a usuarios autenticados, y qué código HTTP devuelve GitHub al excederlo?"
)

## 7. Fase 3: Lógica de Agente y Herramientas (*Tools*)

Siguiendo las directrices de la consigna y lo aprendido en la cátedra sobre **Agentes Autónomos y el patrón ReAct (*Reasoning + Acting*)**:
> *"El sistema no solo debe responder preguntas basándose en el RAG, sino que debe contar con al menos una acción o herramienta adicional (por ejemplo: un módulo de cálculo matemático, un formateador de reportes o búsqueda externa).*  
> *Flujo Esperado: El agente recibe la consulta del usuario, evalúa qué herramientas necesita, recupera los fragmentos del corpus mediante el RAG si es necesario, ejecuta la acción complementaria y genera una respuesta final integrada y fundamentada."*

### Herramientas que implementamos para el Agente:
1. **`consultar_documentacion_github(consulta: str)`:** Herramienta RAG que encapsula la base vectorial FAISS para consultar la especificación técnica oficial.
2. **`consultar_repositorio_github(owner: str, repo: str)`:** Herramienta de acción en vivo que realiza peticiones HTTP reales a la API pública de GitHub (`https://api.github.com/repos/{owner}/{repo}`) para obtener métricas actuales (estrellas, forks, issues abiertos, lenguaje, descripción, etc.).
3. **`calcular_cuota_rate_limit(solicitudes_por_hora: int, es_autenticado: bool)`:** Herramienta analítica para validar el consumo de cuota oficial según las reglas de autenticación (60 vs 5000 req/h).

In [ ]:
# ==============================================================================
# DEFINICIÓN DE HERRAMIENTAS (TOOLS) CON LANGCHAIN (@tool)
# ==============================================================================

@tool
def consultar_documentacion_github(consulta: str) -> str:
    """
    Consulta la documentación técnica oficial de la API REST de GitHub (v3).
    Útil para conocer: endpoints, métodos HTTP (GET, POST, PATCH, PUT, DELETE),
    parámetros requeridos y opcionales, códigos de estado HTTP, cabeceras (headers),
    límites teóricos de rate limit y procedimientos de autenticación.
    Input esperado: una consulta técnica en texto descriptivo.
    """
    docs = retriever.invoke(consulta)
    if not docs:
        return "No se encontraron fragmentos relevantes en la documentación técnica."
    
    return formatear_contexto(docs)


@tool
def consultar_repositorio_github(owner: str, repo: str) -> str:
    """
    Consulta información y métricas EN VIVO de un repositorio público en GitHub
    a través de una llamada real a la API REST pública de GitHub.
    Input esperado:
      - owner: usuario u organización dueña del repositorio (ej: 'facebook', 'octocat').
      - repo: nombre del repositorio (ej: 'react', 'Hello-World').
    Devuelve: estrellas actuales, forks, cantidad de issues abiertos, lenguaje principal,
    descripción y fecha de última actualización.
    """
    url = f"https://api.github.com/repos/{owner}/{repo}"
    headers = {
        "User-Agent": "Laboratorio-IA-TP2-Agent/1.0",
        "Accept": "application/vnd.github+json"
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        
        if response.status_code == 200:
            data = response.json()
            resultado = {
                "repositorio": data.get("full_name"),
                "descripcion": data.get("description", "Sin descripción"),
                "estrellas": data.get("stargazers_count", 0),
                "forks": data.get("forks_count", 0),
                "issues_abiertos": data.get("open_issues_count", 0),
                "lenguaje_principal": data.get("language", "No especificado"),
                "ultima_actualizacion": data.get("updated_at", ""),
                "url_html": data.get("html_url", "")
            }
            return json.dumps(resultado, indent=2, ensure_ascii=False)
        elif response.status_code == 404:
            return f"Error 404: El repositorio '{owner}/{repo}' no fue encontrado en GitHub."
        elif response.status_code == 403:
            return "Error 403: Se alcanzó el límite de solicitudes por hora no autenticadas a la API pública de GitHub."
        else:
            return f"Error HTTP {response.status_code}: {response.text}"
            
    except Exception as e:
        return f"Error de conexión al consultar GitHub API: {str(e)}"


@tool
def calcular_cuota_rate_limit(solicitudes_por_hora: int, es_autenticado: bool = False) -> str:
    """
    Calcula y evalúa el estado del Rate Limit según las reglas oficiales de la API de GitHub.
    Input esperado:
      - solicitudes_por_hora: número estimado de solicitudes por hora que planea realizar el usuario.
      - es_autenticado: True si se utiliza token de autenticación (PAT u OAuth), False si es anónimo.
    Devuelve: límite total permitido, porcentaje de cuota consumido y advertencias si excede el límite.
    """
    limite = 5000 if es_autenticado else 60
    restante = max(0, limite - solicitudes_por_hora)
    porcentaje = min(100.0, (solicitudes_por_hora / limite) * 100)
    
    estado = "DENTRO DEL LÍMITE" if solicitudes_por_hora <= limite else "EXCEDIDO (Error 403 esperado)"
    
    return json.dumps({
        "tipo_usuario": "Autenticado (PAT/OAuth)" if es_autenticado else "No Autenticado (Anónimo)",
        "limite_por_hora": limite,
        "solicitudes_planificadas": solicitudes_por_hora,
        "solicitudes_restantes": restante,
        "porcentaje_consumido": f"{porcentaje:.1f}%",
        "estado": estado
    }, indent=2, ensure_ascii=False)


# Lista de herramientas disponibles para el agente
tools = [
    consultar_documentacion_github,
    consultar_repositorio_github,
    calcular_cuota_rate_limit
]

print(f"✅ Herramientas definidas e inicializadas ({len(tools)} tools):")
for t in tools:
    print(f"  🔧 {t.name}: {t.description.strip()[:80]}...")

## 8. Fase 3: Orquestación del Agente Interactivo (ReAct con LangChain)

Orquestamos el agente conectando el modelo local `qwen2.5:1.5b` con la lista de herramientas.
Para garantizar máxima robustez y compatibilidad con lo enseñado en el Módulo 17 (`Agente_LangChain.ipynb`), implementamos un runtime que soporta tanto `create_agent` (LangGraph) como `create_tool_calling_agent` / `AgentExecutor`.

El agente sigue el ciclo iterativo **ReAct**:
1. **Pensamiento (*Thought*):** Evalúa la intención del usuario y determina qué herramientas necesita.
2. **Acción (*Action*):** Ejecuta la herramienta de RAG para la documentación, la consulta HTTP en vivo para datos del repositorio, o ambas.
3. **Observación (*Observation*):** Recibe el contenido de los fragmentos técnicos o los metadatos JSON devueltos.
4. **Respuesta Final (*Final Answer*):** Sintetiza una respuesta técnica, clara, fundamentada y con trazabilidad de fuentes.

In [ ]:
# ==============================================================================
# ORQUESTACIÓN DEL AGENTE INTERACTIVO
# ==============================================================================

system_prompt_agente = (
    "Eres un Asistente Técnico y Agente Especialista en GitHub y su API REST (v3).\n"
    "Cuentas con herramientas para responder con máxima precisión:\n"
    "1. 'consultar_documentacion_github': Úsala para obtener especificaciones de endpoints, métodos HTTP, cabeceras, parámetros JSON y reglas teóricas de la API según el corpus oficial.\n"
    "2. 'consultar_repositorio_github': Úsala cuando te pregunten por datos en vivo de un repositorio real (estrellas, forks, issues abiertos, lenguaje, etc.).\n"
    "3. 'calcular_cuota_rate_limit': Úsala para verificar y calcular el consumo de cuotas de peticiones.\n\n"
    "Instrucciones obligatorias:\n"
    "- Si la pregunta pide especificaciones de la documentación y TAMBIÉN datos en vivo de un repositorio, debes usar AMBAS herramientas e integrar ambos resultados.\n"
    "- Incluye siempre citas a los documentos de origen cuando uses información del corpus (ej: [Fuente: 03_issues_y_etiquetas.md]).\n"
    "- Sé estructurado, profesional y conciso en tus explicaciones."
)

# Orquestación compatible del agente
try:
    from langchain.agents import create_agent
    agent_executor = create_agent(
        model=llm,
        tools=tools,
        system_prompt=system_prompt_agente
    )
    modo_agente = "create_agent (LangGraph)"
except (ImportError, Exception):
    from langchain.agents import create_tool_calling_agent, AgentExecutor
    from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

    prompt_agente = ChatPromptTemplate.from_messages([
        ("system", system_prompt_agente),
        ("human", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad", optional=True)
    ])
    
    agent = create_tool_calling_agent(llm, tools, prompt_agente)
    agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
    modo_agente = "create_tool_calling_agent + AgentExecutor"

print(f"✅ Agente interactivo orquestado correctamente ({modo_agente}).")


def ejecutar_consulta_agente(pregunta: str) -> str:
    """Función unificada para invocar al agente y presentar la interacción."""
    print("=" * 80)
    print(f"🧑‍💻 USUARIO: {pregunta}")
    print("=" * 80)
    
    try:
        if modo_agente.startswith("create_agent"):
            salida = agent_executor.invoke({"messages": [{"role": "user", "content": pregunta}]})
            respuesta_texto = salida["messages"][-1].content
        else:
            salida = agent_executor.invoke({"input": pregunta})
            respuesta_texto = salida.get("output", str(salida))
    except Exception as e:
        respuesta_texto = f"Error durante la ejecución del agente: {str(e)}"
    
    print(f"🤖 AGENTE INTERACTIVO:\n{respuesta_texto}\n")
    return respuesta_texto

## 9. Pruebas de Interacción del Flujo Completo (Entregable del Laboratorio)

La consigna exige demostrar el funcionamiento del agente mediante:
> *"Un ejemplo real de interacción que muestre al menos 3 preguntas y sus respectivas respuestas correctas."*  
> *"El agente recibe la consulta del usuario, evalúa qué herramientas necesita, recupera los fragmentos del corpus mediante el RAG si es necesario, ejecuta la acción complementaria y genera una respuesta final integrada y fundamentada."*

Evaluamos los 3 casos de uso arquitectónicos:
* **Caso 1 (Solo RAG):** Consulta puramente teórica sobre especificaciones de la documentación (cabeceras requeridas y códigos de error por omisión de `User-Agent`).
* **Caso 2 (Solo Tool Externa en Vivo):** Consulta sobre métricas actuales y reales de un repositorio en GitHub (`facebook/react`).
* **Caso 3 (Flujo Combinado RAG + Tool Externa):** Consulta compleja que requiere saber cómo crear un issue según la documentación oficial y al mismo tiempo averiguar cuántos issues abiertos tiene hoy dicho repositorio en vivo.

In [ ]:
# ==============================================================================
# EJECUCIÓN DE LAS 3 CONSULTAS DE PRUEBA (ENTREGABLE DEL LABORATORIO)
# ==============================================================================

# CASO 1: Consulta que requiere exclusivamente la herramienta de RAG
print("\n>>> CASO 1: Consulta de Especificación Técnica (Solo RAG)")
resp_caso1 = ejecutar_consulta_agente(
    "¿Cuáles son los encabezados HTTP obligatorios para hacer peticiones a la API REST de GitHub según la documentación oficial y qué código de estado devuelve si falta el User-Agent?"
)

# CASO 2: Consulta que requiere exclusivamente la herramienta en vivo
print("\n>>> CASO 2: Consulta de Métricas Actuales en Vivo (Solo Tool Externa)")
resp_caso2 = ejecutar_consulta_agente(
    "¿Cuántas estrellas, forks e issues abiertos tiene actualmente el repositorio facebook/react y cuál es su lenguaje principal?"
)

# CASO 3: Flujo Completo Combinado (RAG + Tool Externa en Vivo)
print("\n>>> CASO 3: Flujo Completo Integrado (RAG + Tool Externa en Vivo)")
resp_caso3 = ejecutar_consulta_agente(
    "¿Qué método HTTP y endpoint se utiliza para crear un issue según la documentación técnica oficial de GitHub (menciona los campos obligatorios del cuerpo JSON) y cuántos issues abiertos tiene actualmente el repositorio facebook/react?"
)